In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")


In [2]:
import pandas as pd
from src.utils.general import load_json

In [3]:
ROOT = "./models/text"

In [4]:
models = list(os.listdir(f"{ROOT}/node"))

raw_metrics = dict()

# 1. Separate (Node + Edge)
for model_setting in models:
    setting, model = model_setting.split("_")
    for dataset in os.listdir(f"{ROOT}/node/{model_setting}"):
        for split_type in os.listdir(f"{ROOT}/node/{model_setting}/{dataset}"):
            node_p = f"{ROOT}/node/{model_setting}/{dataset}/{split_type}/test_metrics.json"
            edge_p = f"{ROOT}/edge/{model_setting}/{dataset}/{split_type}/test_metrics.json"
            if os.path.exists(node_p) and os.path.exists(edge_p):
                node_metrics = load_json(node_p)
                edge_metrics = load_json(edge_p)
                raw_metrics[(setting, model, "separate", dataset, split_type)] = {
                    **{("Node", m): v for m, v in node_metrics.items()},
                    **{("Edge", m): v for m, v in edge_metrics.items()},
                }

# 2. Complete
if os.path.exists(f"{ROOT}/complete"):
    for model_setting in os.listdir(f"{ROOT}/complete"):
        setting, model = model_setting.split("_")
        for dataset in os.listdir(f"{ROOT}/complete/{model_setting}"):
            for split_type in os.listdir(f"{ROOT}/complete/{model_setting}/{dataset}"):
                comp_p = f"{ROOT}/complete/{model_setting}/{dataset}/{split_type}/test_metrics.json"
                if os.path.exists(comp_p):
                    comp_metrics = load_json(comp_p)
                    node_metrics = comp_metrics.get("node", {})
                    edge_metrics = comp_metrics.get("rel") or comp_metrics.get("edge", {})
                    raw_metrics[(setting, model, "complete", dataset, split_type)] = {
                        **{("Node", m): v for m, v in node_metrics.items()},
                        **{("Edge", m): v for m, v in edge_metrics.items()},
                    }

df = pd.DataFrame.from_dict(raw_metrics, orient="index")
df.index.names = ["setting", "model", "approach", "dataset", "split"]
df.columns.names = ["task", "metric"]


In [5]:
df

task                                                Node                      \
metric                                                f1 precision    recall   
setting model      approach dataset  split                                     
shared  bert       separate dataset1 inductive  0.190521  0.206893  0.337220   
                                     hybrid     0.090818  0.104089  0.156593   
                            dataset2 inductive  0.379000  0.441073  0.569103   
                                     hybrid     0.287111  0.316734  0.428963   
        bge        separate dataset1 inductive  0.228104  0.255859  0.377867   
                                     hybrid     0.094303  0.107565  0.161148   
                            dataset2 inductive  0.426344  0.433637  0.612313   
                                     hybrid     0.282951  0.320226  0.413036   
        modernbert separate dataset1 inductive  0.169762  0.223910  0.255602   
                                     hybrid     0.078619  0.094323  0.134839   
                            dataset2 inductive  0.282056  0.385911  0.334953   
                                     hybrid     0.234058  0.287527  0.350920   
        roberta    separate dataset1 inductive  0.188396  0.216896  0.275570   
                                     hybrid     0.079103  0.093809  0.121133   
                            dataset2 inductive  0.252635  0.376651  0.434422   
                                     hybrid     0.249164  0.289661  0.377561   
        bert       complete dataset1 inductive  0.320172  0.354010  0.373398   
                                     hybrid     0.042903  0.058688  0.072240   
                            dataset2 inductive  0.243497  0.271327  0.359032   
                                     hybrid     0.148783  0.191314  0.216667   
        bge        complete dataset1 inductive  0.327069  0.363811  0.361279   
                                     hybrid     0.069369  0.086736  0.101791   
                            dataset2 inductive  0.283010  0.289685  0.370862   
                                     hybrid     0.121983  0.162861  0.157835   
        modernbert complete dataset1 inductive  0.235620  0.279294  0.253573   
                                     hybrid     0.063216  0.069302  0.094564   
                            dataset2 inductive  0.272932  0.298490  0.359122   
                                     hybrid     0.088915  0.123518  0.129452   
        roberta    complete dataset1 inductive  0.323617  0.353208  0.364203   
                                     hybrid     0.066370  0.068813  0.105056   
                            dataset2 inductive  0.311099  0.367004  0.432525   
                                     hybrid     0.208015  0.246243  0.250173   

task                                                Edge                      
metric                                                f1 precision    recall  
setting model      approach dataset  split                                    
shared  bert       separate dataset1 inductive  0.138616  0.153922  0.140934  
                                     hybrid     0.033786  0.033404  0.046549  
                            dataset2 inductive  0.083438  0.094881  0.097622  
                                     hybrid     0.067817  0.076538  0.091252  
        bge        separate dataset1 inductive  0.183940  0.215774  0.187041  
                                     hybrid     0.036893  0.038107  0.051403  
                            dataset2 inductive  0.089954  0.106820  0.125794  
                                     hybrid     0.077683  0.100438  0.089564  
        modernbert separate dataset1 inductive  0.095906  0.115667  0.095241  
                                     hybrid     0.024538  0.024539  0.037415  
                            dataset2 inductive  0.051412  0.065499  0.071707  
                                     hybrid     0.047625  0.059561  0.061965  
        robert

# DATASET 1

In [6]:
df.xs(("dataset1", "hybrid"), level=["dataset", "split"]).sort_index()

task                             Node                          Edge            \
metric                             f1 precision    recall        f1 precision   
setting model      approach                                                     
shared  bert       complete  0.042903  0.058688  0.072240  0.036435  0.036460   
                   separate  0.090818  0.104089  0.156593  0.033786  0.033404   
        bge        complete  0.069369  0.086736  0.101791  0.032934  0.034201   
                   separate  0.094303  0.107565  0.161148  0.036893  0.038107   
        modernbert complete  0.063216  0.069302  0.094564  0.023617  0.024269   
                   separate  0.078619  0.094323  0.134839  0.024538  0.024539   
        roberta    complete  0.066370  0.068813  0.105056  0.026006  0.026768   
                   separate  0.079103  0.093809  0.121133  0.032251  0.033089   

task                                   
metric                         recall  
setting model      approach            
shared  bert       complete  0.050342  
                   separate  0.046549  
        bge        complete  0.048335  
                   separate  0.051403  
        modernbert complete  0.036238  
                   separate  0.037415  
        roberta    complete  0.036470  
                   separate  0.046947

In [7]:
df.xs(("dataset1", "inductive"), level=["dataset", "split"]).sort_index()

task                             Node                          Edge            \
metric                             f1 precision    recall        f1 precision   
setting model      approach                                                     
shared  bert       complete  0.320172  0.354010  0.373398  0.139436  0.158034   
                   separate  0.190521  0.206893  0.337220  0.138616  0.153922   
        bge        complete  0.327069  0.363811  0.361279  0.203532  0.233498   
                   separate  0.228104  0.255859  0.377867  0.183940  0.215774   
        modernbert complete  0.235620  0.279294  0.253573  0.116589  0.131045   
                   separate  0.169762  0.223910  0.255602  0.095906  0.115667   
        roberta    complete  0.323617  0.353208  0.364203  0.135767  0.157915   
                   separate  0.188396  0.216896  0.275570  0.148758  0.176263   

task                                   
metric                         recall  
setting model      approach            
shared  bert       complete  0.139834  
                   separate  0.140934  
        bge        complete  0.205019  
                   separate  0.187041  
        modernbert complete  0.116832  
                   separate  0.095241  
        roberta    complete  0.138000  
                   separate  0.148392

# DATASET 2

In [8]:
df.xs(("dataset2", "hybrid"), level=["dataset", "split"]).sort_index()

task                             Node                          Edge            \
metric                             f1 precision    recall        f1 precision   
setting model      approach                                                     
shared  bert       complete  0.148783  0.191314  0.216667  0.055269  0.062849   
                   separate  0.287111  0.316734  0.428963  0.067817  0.076538   
        bge        complete  0.121983  0.162861  0.157835  0.072971  0.089240   
                   separate  0.282951  0.320226  0.413036  0.077683  0.100438   
        modernbert complete  0.088915  0.123518  0.129452  0.049841  0.066705   
                   separate  0.234058  0.287527  0.350920  0.047625  0.059561   
        roberta    complete  0.208015  0.246243  0.250173  0.092670  0.106021   
                   separate  0.249164  0.289661  0.377561  0.082637  0.093478   

task                                   
metric                         recall  
setting model      approach            
shared  bert       complete  0.064931  
                   separate  0.091252  
        bge        complete  0.087217  
                   separate  0.089564  
        modernbert complete  0.059132  
                   separate  0.061965  
        roberta    complete  0.107985  
                   separate  0.101045

In [9]:
df.xs(("dataset2", "inductive"), level=["dataset", "split"]).sort_index()

task                             Node                          Edge            \
metric                             f1 precision    recall        f1 precision   
setting model      approach                                                     
shared  bert       complete  0.243497  0.271327  0.359032  0.061827  0.075051   
                   separate  0.379000  0.441073  0.569103  0.083438  0.094881   
        bge        complete  0.283010  0.289685  0.370862  0.082222  0.100276   
                   separate  0.426344  0.433637  0.612313  0.089954  0.106820   
        modernbert complete  0.272932  0.298490  0.359122  0.042278  0.055621   
                   separate  0.282056  0.385911  0.334953  0.051412  0.065499   
        roberta    complete  0.311099  0.367004  0.432525  0.091390  0.112286   
                   separate  0.252635  0.376651  0.434422  0.090034  0.099758   

task                                   
metric                         recall  
setting model      approach            
shared  bert       complete  0.099631  
                   separate  0.097622  
        bge        complete  0.112045  
                   separate  0.125794  
        modernbert complete  0.054828  
                   separate  0.071707  
        roberta    complete  0.132452  
                   separate  0.141417